# Long-only 2 de 5 ações — notebook modular (v1)

O código está em `src/long_only_v1/`. Este notebook só **chama** as funções e mostra os resultados.

In [ ]:
# Setup
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd

from long_only_v1.config import TICKERS, START_DATE, END_DATE, TRAIN_END, RF_MONTHLY
from long_only_v1.data import download_prices, validate_prices
from long_only_v1 import returns as R, metrics as M, strategy as S, costs as C, robustness as RB, plots as P

P.set_style()
pd.set_option("display.max_columns", 10)

# O notebook fica em notebooks/, os resultados em results/v1/
RESULTS = Path.cwd().parent / "results" / "v1"
FIGS = RESULTS / "figures"
TABLES = RESULTS / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

## 1. Dados

In [ ]:
prices = download_prices(TICKERS, START_DATE, END_DATE)
validate_prices(prices, TICKERS, END_DATE)

print("Primeira data:", prices.index.min().date(), "| Última data:", prices.index.max().date())
display(prices.tail())
P.plot_prices(prices, save_path=FIGS / "01_precos.png")

## 2. Retornos e benchmark

In [ ]:
daily_returns = R.daily_returns(prices)
cumulative_returns = R.growth_of_one(daily_returns)
P.plot_daily_growth(cumulative_returns, save_path=FIGS / "02_crescimento_diario.png")

monthly_prices = R.to_monthly_prices(prices)
monthly_returns = R.to_monthly_returns(monthly_prices)
benchmark_returns = R.equal_weight_benchmark(monthly_returns)

monthly_stock_growth = R.growth_of_one(monthly_returns)
benchmark_growth = R.growth_of_one(benchmark_returns)
P.plot_monthly_growth(monthly_stock_growth, benchmark_growth, save_path=FIGS / "03_acoes_benchmark.png")

## 3. Estratégia (seleção, escolha do peso no treino e carteira final)

In [ ]:
out = S.run_strategy_inverse_vol(prices)
weights = out["weights"]
strategy_returns = out["strategy_returns"]
STRATEGY_LABEL = S.INVERSE_VOL_LABEL

In [ ]:
# Validação da carteira: sempre 2 ações, 100% investido, sem alavancagem
active = weights.loc[weights.sum(axis=1) > 0]
assert np.allclose(active.sum(axis=1), 1.0)
assert ((active > 0).sum(axis=1) == 2).all()
assert (weights >= 0).all().all()
print("Meses investidos:", len(active), "| Meses sem alocação:", len(weights) - len(active))

## 4. Estratégia vs. benchmark

In [ ]:
comparison = pd.concat(
    [strategy_returns.rename("Estrategia"), benchmark_returns.rename("Benchmark")], axis=1
).dropna()
train_comparison, test_comparison = M.split_train_test(comparison, TRAIN_END)
print("Meses de treino:", len(train_comparison), "| Meses de teste:", len(test_comparison))

metrics_train = M.metrics_table(train_comparison, RF_MONTHLY)
metrics_test = M.metrics_table(test_comparison, RF_MONTHLY)
display(metrics_train)
display(metrics_test)
metrics_train.to_csv(TABLES / "metricas_treino.csv")
metrics_test.to_csv(TABLES / "metricas_teste.csv")

P.plot_growth(train_comparison, "Treino: estratégia vs. benchmark", save_path=FIGS / "05_crescimento_treino.png")
P.plot_growth(test_comparison, "Out-of-sample: estratégia vs. benchmark", save_path=FIGS / "06_crescimento_teste.png")
P.plot_drawdown(train_comparison, "Drawdown no treino", save_path=FIGS / "07_drawdown_treino.png")
P.plot_drawdown(test_comparison, "Drawdown out-of-sample", save_path=FIGS / "08_drawdown_teste.png")

## 5. Referência: S&P 500 (SPY)

In [ ]:
spy_prices = download_prices(["SPY"], START_DATE, END_DATE)["SPY"]
assert spy_prices.index.max().to_period("M") == prices.index.max().to_period("M")
spy_returns = R.to_monthly_returns(R.to_monthly_prices(spy_prices))

comparison_all = pd.concat(
    [strategy_returns.rename(STRATEGY_LABEL),
     benchmark_returns.rename("Benchmark 5 ativos"),
     spy_returns.rename("S&P 500 (proxy SPY)")],
    axis=1, join="inner",
).dropna()
comparison_train, comparison_test = M.split_train_test(comparison_all, TRAIN_END)

display(M.metrics_table(comparison_train, RF_MONTHLY))
display(M.metrics_table(comparison_test, RF_MONTHLY))
P.plot_growth(comparison_test, "Out-of-sample: estratégia vs. benchmark vs. S&P 500", save_path=FIGS / "09_teste_com_spy.png")

## 6. Robustez: primeira e segunda metade

In [ ]:
display(RB.compare_halves(comparison_train, "Treino", RF_MONTHLY))
display(RB.compare_halves(comparison_test, "Teste", RF_MONTHLY))

## 7. Turnover e custos de transação

In [ ]:
turnover = C.monthly_turnover(weights, strategy_returns.index)
display(C.turnover_summary(turnover))

cost_table_train = C.evaluate_transaction_costs(strategy_returns, turnover, TRAIN_END, "train", RF_MONTHLY)
cost_table_test = C.evaluate_transaction_costs(strategy_returns, turnover, TRAIN_END, "test", RF_MONTHLY)
print("Custos no treino"); display(cost_table_train)
print("Custos no teste"); display(cost_table_test)
cost_table_train.to_csv(TABLES / "custos_treino.csv")
cost_table_test.to_csv(TABLES / "custos_teste.csv")

## 8. Gráfico final

In [ ]:
strategy_growth = R.growth_of_one(strategy_returns.dropna())
P.plot_monthly_growth(monthly_stock_growth, benchmark_growth, strategy_growth,
                      strategy_label=STRATEGY_LABEL, save_path=FIGS / "10_final.png")